In [0]:
# ============================================================
# NOTEBOOK 2 — Modelagem de Personalização de Ofertas
# Case Técnico iFood 
# Autoria: Ana Clara Aragão Fernandes
# ============================================================
# Input : workspace.default.ifood_final_dataset (76.277 linhas)
# Output: recomendação de oferta por cliente + impacto financeiro
#
# Abordagem: S-Learner LightGBM (Künzel et al., 2019)
#   - Tipo de oferta tratado como feature de tratamento (T_i)
#   - Simulação contrafactual por cliente para cada tipo de oferta
#   - Uplift = P(Y=1|X_i, offer_j) - P(Y=1|Xi, offer_k) para todo k≠j
#
# Referências principais:
#   - Lo (2002): True Lift Model — objetivo correto é maximizar
#     E(Y|X,tratamento) - E(Y|X,controle), não só P(Y=1|X)
#   - Shchetkina & Berman (2024): personalização só gera valor
#     quando há heterogeneidade acionável (crossover entre tratamentos)
# ============================================================

In [0]:
%pip install xgboost lightgbm shap

In [0]:
dbutils.library.restartPython()

In [0]:

# --- Spark / PySpark ---
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.types import *

# --- Manipulação e análise ---
import pandas as pd
import numpy as np

# --- Visualização ---
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# --- Machine Learning (scikit-learn) ---
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.cluster import KMeans

# --- Modelos ---
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb

# --- Estatística (testes de balanceamento) ---
from scipy import stats

# --- Configurações gerais ---

SEED = 220613
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)




In [0]:
final_df = spark.table("workspace.default.ifood_final_dataset")

print("Total de linhas:", final_df.count())
print("Total de colunas:", len(final_df.columns))
final_df.printSchema()

In [0]:
# ============================================================
# CONTENTS — Dataset Final Unificado
# ============================================================

contents = (
    final_df
    .select(
        *[
            F.struct(
                F.lit(c).alias("coluna"),
                F.lit(str(final_df.schema[c].dataType)).alias("tipo"),
                F.count(F.when(F.col(c).isNull(), 1)).alias("nulos"),
                F.countDistinct(c).alias("distintos"),
            )
            for c in final_df.columns
        ]
    )
)

# Mais legível: uma linha por coluna
summary_rows = []
total = final_df.count()

for c in final_df.columns:
    dtype = str(final_df.schema[c].dataType)
    nulos = final_df.filter(F.col(c).isNull()).count()
    distintos = final_df.select(c).distinct().count()
    exemplo = final_df.select(c).dropna().limit(1).collect()
    exemplo_val = exemplo[0][0] if exemplo else None
    summary_rows.append({
        "coluna": c,
        "tipo": dtype,
        "nulos": nulos,
        "pct_nulo": f"{100*nulos/total:.1f}%",
        "distintos": distintos,
        "exemplo": exemplo_val
    })

contents_pd = pd.DataFrame(summary_rows)
contents_pd

In [0]:
final_df.limit(50).toPandas()


## EDA — Validação do Desenho Experimental

Antes de modelar, validamos duas condições estatísticas que sustentam o uso do S-Learner:

**1. Balanceamento covariável** — os grupos que receberam cada tipo de oferta têm perfil similar?
Se sim, diferenças de conversão são atribuíveis ao tipo de oferta, não ao perfil do cliente.
*(Lo, 2002: grupos de tratamento e controle precisam ser comparáveis em Xi)*

**2. Significância da diferença de conversão** — as taxas de conversão entre tipos de oferta
são estatisticamente diferentes, ou podem ser explicadas por acaso?
Se sim, há base estatística para modelar personalização por tipo de oferta.

In [0]:
# ============================================================
# PARTE 1 — FEATURE ENGINEERING
# Referência: Lo (2002), Seção 4.4, passos 5-7
# "Variable reduction: narrow down the list of independent 
#  variables Xi before multiplying by Ti"
# ============================================================

# Carrega a base final (gerada no notebook 1)
final_df = spark.table("workspace.default.ifood_final_dataset")

# Converte pra pandas — base tem 76k linhas, cabe em memória
# (LightGBM e sklearn não operam em DataFrames Spark nativamente)
df = final_df.toPandas()

print(f"Shape: {df.shape}")
df.head()

In [0]:
# ============================================================
# PARTE 1.1 — FEATURE ENGINEERING: variáveis de perfil do cliente
#
# Lo (2002) chama de Xi: "vetor de p variáveis independentes que representam características individuais (idade, renda, comportamento transacional passado)"
#
# Decisões tomadas e justificativas:
# - age: mantida com nulos (LightGBM trata nulos nativamente, sem imputação — reduz viés de imputação)
# - age_missing: flag explícita (clientes com cadastro incompleto convertem ~3x menos — achado da EDA, não deve ficar implícito no nulo)
# - credit_card_limit: proxy de poder aquisitivo, mantida com nulos
# - gender: codificada como ordinal simples (M/F/O/null → 0/1/2/3) - LightGBM aceita variáveis categóricas — não precisamos de dummies
# - registered_on_date: convertida em "dias de cadastro até o início do teste" (tempo de casa) — mais informativo que a data bruta - aproximada pela ultima data de cadastro da base
# - channel_email: DROPADA (variância zero — toda oferta é enviada por email, confirmado na EDA: distintos=1)
# ============================================================

import pandas as pd
import numpy as np
from datetime import date

# Data de referência = início do teste (time_since_test_start = 0)
# Não temos a data absoluta, mas o que importa é o tempo relativo
# de cadastro — estimamos como dias entre registered_on_date e a data máxima observada na base (proxy do início do experimento)
DATA_REFERENCIA = pd.to_datetime(df["registered_on_date"]).max()

df["dias_de_cadastro"] = (
    pd.to_datetime(df["registered_on_date"]) - DATA_REFERENCIA
).dt.days

# Codifica gender como numérico (LightGBM aceita categoria, mas
# é mais seguro converter explicitamente pro sklearn pipeline)
gender_map = {"M": 0, "F": 1, "O": 2}
df["gender_encoded"] = df["gender"].map(gender_map).fillna(3)  # 3 = não informado

# Lista final de features Xi (perfil do cliente)
FEATURES_CLIENTE = [
    "age",              # idade (nulos mantidos — LightGBM trata nativamente)
    "age_missing",      # flag: cadastro incompleto (sinal forte de baixo engajamento)
    "credit_card_limit",# proxy de poder aquisitivo
    "gender_encoded",   # gênero codificado
    "dias_de_cadastro", # tempo de relacionamento com a plataforma
]

# Features de metadados da oferta (além do offer_type, que é o "Ti")
FEATURES_OFERTA = [
    "min_value",        # valor mínimo pra ativar a oferta
    "discount_value",   # valor do desconto/recompensa
    "duration",         # janela de validade em dias
    "channel_mobile",   # se a oferta foi enviada por mobile
    "channel_web",      # se a oferta foi enviada por web
    "channel_social",   # se a oferta foi enviada por social media
    # channel_email DROPADO: variância zero (toda oferta é enviada por email)
]

# Ti: variável de tratamento (Lo, 2002, equação 4)
# No S-Learner, Ti entra como feature regular — não como dummy separada
# Codificamos como ordinal: LightGBM vai aprender internamente
# as interações Xi*Ti que Lo (2002) cria explicitamente em logistic regression
TRATAMENTO = "offer_type"
offer_type_map = {"bogo": 0, "discount": 1, "informational": 2}
df["offer_type_encoded"] = df[TRATAMENTO].map(offer_type_map)

# Feature set completo = Xi + Ti (S-Learner: tudo num único modelo)
FEATURES = FEATURES_CLIENTE + FEATURES_OFERTA + ["offer_type_encoded"]

# Variável resposta: Yi (Lo, 2002)
TARGET = "target_engaged_conversion"

print("Features selecionadas:", FEATURES)
print("Target:", TARGET)
print(f"\nDistribuição do target:\n{df[TARGET].value_counts(normalize=True).round(3)}")

In [0]:
# ============================================================
# PARTE 1.2 — SPLIT TREINO/TESTE
#
# Lo (2002), Seção 4.4: "Divide the data set into training and hold-out samples" — validação no holdout é o que garante que o modelo generalize para  
# clientes novos, não apenas memorize o padrão de quem já recebeu oferta.
#
# DECISÃO CRÍTICA: split por customer_id, não por linha
# Por quê isso importa aqui:
# Um mesmo cliente aparece múltiplas vezes no dataset (uma linha por instância de oferta recebida — média de ~4,5 linhas/cliente).
# Se fizermos split aleatório por linha, o modelo vê o cliente 78afa995 no treino (com oferta BOGO) e no teste (com oferta discount) — isso é considerado data leakage: o modelo aprende o comportamento específico do cliente durante o treino e "trapaceia" no teste.
# Split por cliente garante que cada cliente está em UM lado apenas.
# Referência: prática padrão em modelos de CRM — ver Radcliffe (2007), "Using control groups to target on predicted lift"
# ============================================================

from sklearn.model_selection import train_test_split

# Lista de clientes únicos
clientes_unicos = df["customer_id"].unique()

# Split 80/20 por cliente, estratificado pelo target médio por cliente (garantia que a proporção de clientes "conversores" é igual em treino e teste)

clientes_train, clientes_test = train_test_split(
    clientes_unicos,
    test_size=0.2,
    random_state=SEED
)

df_train = df[df["customer_id"].isin(clientes_train)].copy()
df_test  = df[df["customer_id"].isin(clientes_test)].copy()

X_train = df_train[FEATURES]
y_train = df_train[TARGET]
X_test  = df_test[FEATURES]
y_test  = df_test[TARGET]

print(f"Treino: {len(df_train)} linhas | {df_train['customer_id'].nunique()} clientes únicos")
print(f"Teste:  {len(df_test)} linhas  | {df_test['customer_id'].nunique()} clientes únicos")
print(f"\nDistribuição do target no treino: {y_train.mean():.3f}")
print(f"Distribuição do target no teste:  {y_test.mean():.3f}")
print(f"\nDistribuição de offer_type no treino:\n{df_train['offer_type'].value_counts(normalize=True).round(3)}")
print(f"\nDistribuição de offer_type no teste:\n{df_test['offer_type'].value_counts(normalize=True).round(3)}")

In [0]:
# ============================================================
# EDA — PARTE 1: BALANCEAMENTO COVARIÁVEL
#
# Lo (2002), Seção 4.4: grupos precisam ser comparáveis em Xi
# Teste de Kruskal-Wallis: não-paramétrico, não assume normalidade
# H0: distribuições iguais entre bogo, discount e informational
# Rejeitamos H0 se p < 0.05 → grupos desbalanceados → viés potencial
# ============================================================

from scipy import stats
import numpy as np

grupos = {
    offer: df[df["offer_type"] == offer]
    for offer in ["bogo", "discount", "informational"]
}

features_balanceamento = {
    "age": "Idade",
    "credit_card_limit": "Limite do Cartão (R$)",
    "dias_de_cadastro": "Dias de Cadastro"
}

fig, axes = plt.subplots(len(features_balanceamento), 3, figsize=(16, 12))

resultados_kruskal = {}

for i, (feat, label) in enumerate(features_balanceamento.items()):
    amostras = [g[feat].dropna().values for g in grupos.values()]

    stat, pval = stats.kruskal(*amostras)
    resultados_kruskal[feat] = {"H": stat, "p-value": pval}

    for j, (offer, grupo) in enumerate(grupos.items()):
        axes[i, j].hist(
            grupo[feat].dropna(),
            bins=30,
            color=["steelblue", "seagreen", "coral"][j],
            edgecolor="white", alpha=0.8
        )
        axes[i, j].axvline(
            grupo[feat].dropna().mean(),
            color="red", linestyle="--",
            label=f"Média: {grupo[feat].dropna().mean():.1f}"
        )
        axes[i, j].set_title(f"{label} | {offer}", fontsize=9)
        axes[i, j].legend(fontsize=8)
        if j == 0:
            axes[i, j].set_ylabel("Frequência")

plt.suptitle(
    "Balanceamento Covariável por Tipo de Oferta\n"
    "(distribuições similares = aleatorização bem executada)",
    y=1.02, fontsize=12
)
plt.tight_layout()
plt.show()

print("\nTeste de Kruskal-Wallis:")
print(f"{'Feature':<25} {'H':>10} {'p-value':>12} {'Resultado':>25}")
print("-" * 75)
for feat, res in resultados_kruskal.items():
    resultado = "✓ Balanceado (p>0.05)" if res["p-value"] > 0.05 else "✗ Desbalanceado (p≤0.05)"
    print(f"{feat:<25} {res['H']:>10.3f} {res['p-value']:>12.4f} {resultado:>25}")

In [0]:
# ============================================================
# EDA — PARTE 2: SIGNIFICÂNCIA DA DIFERENÇA DE CONVERSÃO
#
# Qui-quadrado de independência:
# H0: taxa de conversão independente do tipo de oferta
# H1: pelo menos um tipo tem taxa diferente
# Rejeitamos H0 se p < 0.05 → há base estatística pra personalizar
#
# IC de Wilson (95%): mais robusto que normal para proporções
# — mostra se os intervalos se sobrepõem entre os tipos de oferta
# ============================================================

from scipy.stats import chi2_contingency, norm
import pandas as pd

# Tabela de contingência
tabela = pd.crosstab(
    df["offer_type"],
    df["target_engaged_conversion"],
    margins=True
)
print("Tabela de Contingência — Tipo de Oferta × Conversão:")
print(tabela)

# Qui-quadrado
chi2, pval, dof, _ = chi2_contingency(tabela.iloc[:-1, :-1])

print(f"\nTeste Qui-Quadrado de Independência:")
print(f"  χ²      = {chi2:.4f}")
print(f"  gl      = {dof}")
print(f"  p-value = {pval:.6f}")
print(f"  {'✓ Rejeitamos H0 (p<0.05)' if pval < 0.05 else '✗ Não rejeitamos H0 (p≥0.05)'}")
if pval < 0.05:
    print(f"  Conclusão: diferenças de conversão entre tipos de oferta são")
    print(f"  estatisticamente significativas — há base para personalizar.")

# IC de Wilson por tipo de oferta
z = norm.ppf(0.975)
print(f"\nTaxa de conversão por tipo com IC 95% (Wilson):")
print(f"{'Tipo':<15} {'Taxa':>8} {'IC inf':>10} {'IC sup':>10} {'N':>8}")
print("-" * 55)

for offer in ["bogo", "discount", "informational"]:
    grupo = df[df["offer_type"] == offer]
    n = len(grupo)
    p = grupo["target_engaged_conversion"].mean()
    denom = 1 + z**2 / n
    centro = (p + z**2 / (2*n)) / denom
    margem = (z * np.sqrt(p*(1-p)/n + z**2/(4*n**2))) / denom
    print(f"{offer:<15} {p:>8.3f} {centro-margem:>10.3f} {centro+margem:>10.3f} {n:>8,}")

In [0]:
# ============================================================
# PARTE 2 — TREINAMENTO DO S-LEARNER
#
# Shchetkina & Berman (2024), EC '24:
# "In the S-Learner version, the intervention indicator is treated as a regular feature fed into the algorithm, and a single model f is trained for all 
# observations: Yi = f(Xi, Ti)"
#
# Lo (2002), equação (6): "E(Yi|Xi) = f(Xi, Ti, Xi*Ti)"
# No LightGBM, as interações Xi*Ti que Lo cria explicitamente em regressão logística são aprendidas automaticamente pelas árvores de decisão — o modelo descobre sozinho quais features de perfil interagem com o tipo de oferta para predizer conversão.
#
# Vantagens de usar o LightGBM neste case:
# - Trata nulos nativamente (age, credit_card_limit sem imputação)
# - Eficiente em datasets tabulares de escala média (76k linhas)
# - Produz probabilidades calibradas (necessário para calcular uplift)
# - Permite extração de feature importance (interpretabilidade para
#   a apresentação de negócio)
# ============================================================

import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

# Parâmetros do modelo
# Nota: não otimizamos hiperparâmetros via grid search aqui
# (escopo do case é demonstrar a metodologia, não maximizar AUC).
# Em produção, usaríamos Optuna ou similar para tuning.
params = {
    "objective": "binary",          # classificação binária: converteu ou não
    "metric": "auc",                # métrica de otimização interna
    "learning_rate": 0.05,          # taxa de aprendizado conservadora
    "num_leaves": 31,               # complexidade do modelo (default LightGBM)
    "min_child_samples": 50,        # mínimo de amostras por folha — evita overfitting
                                    # em subgrupos pequenos de offer_type
    "feature_fraction": 0.8,        # subsample de features por árvore (regularização)
    "bagging_fraction": 0.8,        # subsample de linhas por árvore (regularização)
    "bagging_freq": 5,
    "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),
                                    # compensa leve desbalanceamento 55/45
    "random_state": SEED,
    "verbose": -1,                  # suprime logs de treinamento
}

# Dataset LightGBM nativo (mais eficiente que sklearn API para grandes bases)
dtrain = lgb.Dataset(X_train, label=y_train)
dvalid = lgb.Dataset(X_test,  label=y_test, reference=dtrain)

# Treino com early stopping: para quando AUC no teste não melhora
# por 50 rounds consecutivos — evita overfitting sem precisar definir
# n_estimators fixo a priori
callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=100)  # imprime AUC a cada 100 rounds
]

model = lgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    valid_sets=[dtrain, dvalid],
    valid_names=["treino", "teste"],
    callbacks=callbacks,
)

print(f"\nMelhor iteração: {model.best_iteration}")
print(f"Melhor AUC no teste: {model.best_score['teste']['auc']:.4f}")

In [0]:
# ============================================================
# PARTE 3 — AVALIAÇÃO DO MODELO
#
# Lo (2002), Seção 4.4, passos de validação:
# "For each individual in the hold-out sample, compute the 
#  predicted values of expected Yi for both treatment and control"
#
# Shchetkina & Berman (2024):
# "The value of targeting is determined by actionable heterogeneity
#  — crossovers between interventions when individuals are ranked
#  by their treatment effects"
#
# Métricas escolhidas:
# - AUC-ROC: capacidade de ranquear conversores acima de não-conversores
#   (métrica padrão de uplift modeling — ver Radcliffe & Surry, 2011)
# - Average Precision (PR-AUC): mais informativa que ROC quando o
#   objetivo é identificar os TOP clientes pra receber oferta
#   (análogo ao Qini score de Radcliffe, 2011)
# - Calibration: as probabilidades precisam ser confiáveis como
#   estimativas reais de P(Y=1) — não apenas rankings — porque
#   vamos subtrair probabilidades entre ofertas para calcular uplift
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.calibration import calibration_curve

# Predições no conjunto de teste
y_pred_proba = model.predict(X_test)

print("=" * 50)
print("MÉTRICAS DE AVALIAÇÃO — Conjunto de Teste")
print("=" * 50)
print(f"AUC-ROC:           {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"Average Precision: {average_precision_score(y_test, y_pred_proba):.4f}")
print(f"Baseline (random): {y_test.mean():.4f}")

# ---- Figura 1: Curvas ROC e PR lado a lado ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

RocCurveDisplay.from_predictions(
    y_test, y_pred_proba, ax=axes[0], name="S-Learner LightGBM"
)
axes[0].plot([0,1],[0,1], "k--", label="Random (AUC=0.50)")
axes[0].set_title("Curva ROC — Teste de Conversão de Oferta")
axes[0].legend()

PrecisionRecallDisplay.from_predictions(
    y_test, y_pred_proba, ax=axes[1], name="S-Learner LightGBM"
)
axes[1].axhline(y=y_test.mean(), color="k", linestyle="--", label=f"Baseline ({y_test.mean():.2f})")
axes[1].set_title("Curva Precision-Recall — Teste de Conversão de Oferta")
axes[1].legend()

plt.tight_layout()
plt.show()

# ---- Figura 2: Calibração das probabilidades ----
# Shchetkina & Berman (2024): para calcular uplift como diferença
# de probabilidades entre ofertas, as probabilidades precisam ser
# bem calibradas — não apenas rankings
fig, ax = plt.subplots(figsize=(7, 5))

fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test, y_pred_proba, n_bins=10
)

ax.plot(mean_predicted_value, fraction_of_positives, "s-", label="S-Learner LightGBM")
ax.plot([0, 1], [0, 1], "k--", label="Calibração perfeita")
ax.set_xlabel("Probabilidade predita média")
ax.set_ylabel("Fração de positivos observados")
ax.set_title("Calibração do Modelo\n(quanto as probabilidades refletem a realidade)")
ax.legend()
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# PARTE 4 — SIMULAÇÃO DE UPLIFT POR TIPO DE OFERTA
#
# Shchetkina & Berman (2024), equação central:
# "π_S-XGB(Xi) = argmax_a f̂(Xi, a)"
# Para cada cliente i, simulamos a predição do modelo para cada
# tipo de oferta a ∈ {bogo, discount, informational}, mantendo
# todas as outras features constantes (ceteris paribus).
# A diferença entre as predições é o uplift incremental de cada oferta.
#
# Lo (2002), equação (5):
# "P(treat) - P(control) = true lift individual"
# Aqui, o "controle implícito" é a média das probabilidades
# preditas — não temos grupo controle formal (limitação documentada),
# mas a comparação RELATIVA entre ofertas já é suficiente para
# a decisão de QUAL oferta enviar (não para calcular o uplift absoluto
# em relação a não enviar nenhuma oferta).
# ============================================================

# Usamos o conjunto de TESTE para a simulação
# (clientes que o modelo nunca viu durante o treino)
X_sim = df_test[FEATURES].copy()

# Médias dos metadados de oferta por tipo — usamos a média real
# de cada tipo de oferta nos dados (não valores fixos arbitrários)
# para que a simulação seja fiel às características reais das ofertas
offer_profiles = df_test.groupby("offer_type")[FEATURES_OFERTA].mean()
print("Perfis médios de oferta usados na simulação:")
print(offer_profiles.round(2))

In [0]:
# Simula P(Y=1 | Xi_cliente, offer_type=j) para cada j
resultados_uplift = df_test[["customer_id", "offer_type"]].copy()

for offer_name, offer_code in offer_type_map.items():
    # Cria uma cópia do dataset de teste onde TODOS os clientes
    # recebem o mesmo tipo de oferta j (ceteris paribus)
    X_counterfactual = X_sim.copy()
    X_counterfactual["offer_type_encoded"] = offer_code

    # Substitui os metadados da oferta pelo perfil médio desse tipo
    for feat in FEATURES_OFERTA:
        X_counterfactual[feat] = offer_profiles.loc[offer_name, feat]

    # Prediz P(Y=1) para esse cenário contrafactual
    resultados_uplift[f"p_conv_{offer_name}"] = model.predict(X_counterfactual)

# Calcula o uplift de cada oferta em relação à média das outras
# (proxy do "true lift" de Lo (2002) sem grupo controle formal)
resultados_uplift["p_conv_media"] = resultados_uplift[
    [f"p_conv_{o}" for o in offer_type_map]
].mean(axis=1)

for offer_name in offer_type_map:
    resultados_uplift[f"uplift_{offer_name}"] = (
        resultados_uplift[f"p_conv_{offer_name}"]
        - resultados_uplift["p_conv_media"]
    )

# Recomendação final: oferta com maior P(Y=1) para cada cliente
resultados_uplift["oferta_recomendada"] = resultados_uplift[
    [f"p_conv_{o}" for o in offer_type_map]
].idxmax(axis=1).str.replace("p_conv_", "")

print("\nDistribuição de ofertas recomendadas:")
print(resultados_uplift["oferta_recomendada"].value_counts(normalize=True).round(3))

print("\nSample de resultados:")
resultados_uplift[[
    "customer_id",
    "p_conv_bogo", "p_conv_discount", "p_conv_informational",
    "uplift_bogo", "uplift_discount", "uplift_informational",
    "oferta_recomendada"
]].head(10)

In [0]:
# ============================================================
# PARTE 4.1 — DIAGNÓSTICO: por que BOGO nunca é recomendado?
#
# Hipótese 1: o perfil médio de metadados do bogo é penalizado
#             pelo modelo independente do perfil do cliente
# Hipótese 2: bogo tem performance inferior em TODOS os segmentos
#             (sem crossover com outros tipos)
# Hipótese 3: artefato da simulação com perfil médio — se usarmos
#             os metadados exatos de cada oferta bogo individualmente,
#             o resultado pode mudar
#
# Referência: Shchetkina & Berman (2024)
# "A third force that lowers the value from targeting is high variance
#  in average outcomes because every targeting policy needs to beat
#  the best uniform treatment"
# ============================================================

# Distribuição das probabilidades preditas por tipo de oferta
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, offer in enumerate(["bogo", "discount", "informational"]):
    axes[i].hist(
        resultados_uplift[f"p_conv_{offer}"],
        bins=50,
        color=["steelblue", "seagreen", "coral"][i],
        edgecolor="white",
        alpha=0.8
    )
    axes[i].axvline(
        resultados_uplift[f"p_conv_{offer}"].mean(),
        color="red", linestyle="--",
        label=f"Média: {resultados_uplift[f'p_conv_{offer}'].mean():.3f}"
    )
    axes[i].set_title(f"P(conversão | {offer})")
    axes[i].set_xlabel("Probabilidade predita")
    axes[i].set_ylabel("Frequência")
    axes[i].legend()

plt.suptitle("Distribuição de P(Y=1) por tipo de oferta — conjunto de teste", y=1.02)
plt.tight_layout()
plt.show()

# Médias por tipo
print("Probabilidade média de conversão por tipo de oferta (simulação contrafactual):")
for offer in ["bogo", "discount", "informational"]:
    media = resultados_uplift[f"p_conv_{offer}"].mean()
    print(f"  {offer:15s}: {media:.4f}")

# Verifica se existe ALGUM cliente onde bogo > discount e bogo > informational
bogo_wins = (
    (resultados_uplift["p_conv_bogo"] > resultados_uplift["p_conv_discount"]) &
    (resultados_uplift["p_conv_bogo"] > resultados_uplift["p_conv_informational"])
).sum()
print(f"\nClientes onde BOGO é a melhor opção: {bogo_wins} ({bogo_wins/len(resultados_uplift)*100:.1f}%)")
print(f"Clientes onde DISCOUNT é a melhor opção: {(resultados_uplift['oferta_recomendada']=='discount').sum()}")
print(f"Clientes onde INFORMATIONAL é a melhor opção: {(resultados_uplift['oferta_recomendada']=='informational').sum()}")

In [0]:
# ============================================================
# PARTE 5 — INTERPRETABILIDADE: FEATURE IMPORTANCE
#
# Lo (2002), Seção 4.4, passo 5:
# "Variable reduction: identify which Xi most strongly predict
#  the differential effect between treatment and control"
#
# No S-Learner com LightGBM, a importância das features revela
# quais características do cliente e da oferta mais influenciam
# P(Y=1 | Xi, Ti) — incluindo implicitamente as interações Xi*Ti
# que Lo (2002) modela explicitamente em regressão logística.
#
# Usamos dois tipos de importância:
# - Split: quantas vezes a feature foi usada para dividir uma folha
#   (mede frequência de uso — pode ser dominado por features com
#    muitas categorias)
# - Gain: ganho médio de informação por split da feature
#   (mede impacto real na redução de impureza — métrica preferida)
# ============================================================

import shap
# Calcula SHAP values no conjunto de teste
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Feature importance por Gain (métrica preferida)
lgb.plot_importance(
    model,
    ax=axes[0],
    importance_type="gain",
    title="Feature Importance — Gain\n(impacto real na redução de impureza)",
    xlabel="Ganho médio por split",
)

# Feature importance por Split (frequência de uso)
lgb.plot_importance(
    model,
    ax=axes[1],
    importance_type="split",
    title="Feature Importance — Split\n(frequência de uso nas árvores)",
    xlabel="Número de splits",
)

plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# PARTE 5.1 — SHAP VALUES
#
# Lundberg & Lee (2017) — "A Unified Approach to Interpreting 
# Model Predictions", NeurIPS 2017
#
# SHAP decompõe a predição individual em contribuições por feature,
# revelando não só QUAIS features importam mas em QUE DIREÇÃO:
# - Cor vermelha = valor alto da feature → aumenta P(Y=1)
# - Cor azul = valor baixo da feature → diminui P(Y=1)
# - Posição no eixo X = magnitude da contribuição
#
# Isso responde perguntas como:
# "Clientes mais velhos convertem mais ou menos?"
# "Limite de cartão alto ajuda ou atrapalha a conversão?"
# — que são exatamente as perguntas que o iFood faria em produção
# ============================================================

plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=FEATURES,
    show=False,
    plot_type="dot"
)
plt.title(
    "SHAP Summary Plot — Contribuição de cada feature para P(Y=1)\n"
    "Vermelho = valor alto da feature | Azul = valor baixo\n"
    "Eixo X: impacto no log-odds de conversão",
    fontsize=10
)
plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# PARTE 6 — IMPACTO DE NEGÓCIO
#
# Lo (2002), Tabela 1 e Seção 2:
# "The appropriate measure of the gain due to the treatment is
#  (A-B) - (C-D) — true lift"
#
# Shchetkina & Berman (2024):
# "The value of targeting needs to beat the best uniform treatment"
#
# Aqui calculamos:
# 1. Baseline: taxa de conversão se mandássemos a mesma oferta
#    pra todo mundo (melhor oferta uniforme = discount, 47.6%)
# 2. Estratégia personalizada: taxa de conversão esperada
#    mandando a oferta recomendada pelo modelo pra cada cliente
# 3. Custo operacional: quanto gastamos em cupom pra quem
#    já ia converter de qualquer jeito (Sure Things de Lo, 2002)
# ============================================================

# Taxa de conversão real no conjunto de teste (observada)
taxa_real = y_test.mean()

# Baseline: melhor oferta uniforme (discount, maior taxa histórica)
taxa_baseline_discount = df_test[df_test["offer_type"] == "discount"][TARGET].mean()
taxa_baseline_bogo = df_test[df_test["offer_type"] == "bogo"][TARGET].mean()
taxa_baseline_informational = df_test[df_test["offer_type"] == "informational"][TARGET].mean()

print("=" * 55)
print("TAXAS DE CONVERSÃO OBSERVADAS — Conjunto de Teste")
print("=" * 55)
print(f"Geral (todas as ofertas):   {taxa_real:.3f} ({taxa_real*100:.1f}%)")
print(f"Discount (melhor uniforme): {taxa_baseline_discount:.3f} ({taxa_baseline_discount*100:.1f}%)")
print(f"BOGO:                       {taxa_baseline_bogo:.3f} ({taxa_baseline_bogo*100:.1f}%)")
print(f"Informational:              {taxa_baseline_informational:.3f} ({taxa_baseline_informational*100:.1f}%)")

# Taxa de conversão esperada pela estratégia personalizada
# Para cada cliente no teste, pegamos a P(Y=1) da oferta recomendada
resultados_uplift_indexed = resultados_uplift.set_index(
    df_test.index
) if len(resultados_uplift) == len(df_test) else resultados_uplift

# Alinha pelo índice do df_test
df_test_avaliacao = df_test.copy()
df_test_avaliacao["oferta_recomendada"] = resultados_uplift["oferta_recomendada"].values
df_test_avaliacao["p_conv_recomendada"] = resultados_uplift.apply(
    lambda r: r[f"p_conv_{r['oferta_recomendada']}"], axis=1
).values

taxa_personalizada_esperada = df_test_avaliacao["p_conv_recomendada"].mean()

print(f"\n{'=' * 55}")
print("COMPARAÇÃO: BASELINE vs PERSONALIZAÇÃO")
print("=" * 55)
print(f"Melhor estratégia uniforme (discount): {taxa_baseline_discount*100:.1f}%")
print(f"Estratégia personalizada (esperada):   {taxa_personalizada_esperada*100:.1f}%")
print(f"Ganho absoluto:                        +{(taxa_personalizada_esperada - taxa_baseline_discount)*100:.1f} p.p.")
print(f"Ganho relativo:                        +{((taxa_personalizada_esperada/taxa_baseline_discount)-1)*100:.1f}%")

# Custo operacional: estimativa de desperdício de cupom
# "Sure Things" = clientes que convertem com qualquer oferta
# (P(Y=1|bogo) > 0.7 AND P(Y=1|discount) > 0.7 AND P(Y=1|informational) > 0.7)
# Threshold de 0.7: conservador, mas defensável como "alta propensão independente"
THRESHOLD_SURE_THING = 0.7

sure_things = (
    (resultados_uplift["p_conv_bogo"] > THRESHOLD_SURE_THING) &
    (resultados_uplift["p_conv_discount"] > THRESHOLD_SURE_THING) &
    (resultados_uplift["p_conv_informational"] > THRESHOLD_SURE_THING)
).sum()

total_test = len(resultados_uplift)
pct_sure_things = sure_things / total_test

print(f"\n{'=' * 55}")
print("CUSTO OPERACIONAL — Sure Things (Lo, 2002)")
print("=" * 55)
print(f"Threshold usado: P(Y=1) > {THRESHOLD_SURE_THING} em TODOS os tipos de oferta")
print(f"Sure Things identificados: {sure_things} clientes ({pct_sure_things*100:.1f}%)")
print(f"Interpretação: {pct_sure_things*100:.1f}% dos clientes converteriam")
print(f"independente da oferta — cupom enviado a eles é custo puro,")
print(f"sem incremento de receita.")
print(f"\nSe o valor médio do desconto (discount) é R${df_test['discount_value'].mean():.2f},")
print(f"o custo evitável por rodada de campanha seria:")
print(f"R${sure_things * df_test['discount_value'].mean():.2f} (sure things × valor médio do desconto)")

In [0]:
# ============================================================
# PARTE 6.1 — ESTIMATIVA DE IMPACTO CORRIGIDA
#
# Lo (2002), Tabela 1:
# Comparação correta é sempre observado vs observado, ou predito vs predito — nunca misturar os dois.
#
# Abordagem correta aqui: comparar a taxa de conversão OBSERVADA dos clientes que receberam a oferta que o modelo recomendaria para eles vs os que receberam uma oferta diferente.
# Isso usa apenas dados reais, sem depender das probabilidades preditas como proxy de resultado.
# ============================================================

# Para cada cliente no teste, verifica se a oferta que ele REALMENTE recebeu bate com a oferta recomendada pelo modelo
df_test_avaliacao["oferta_real"] = df_test["offer_type"].values
df_test_avaliacao["modelo_acertou_oferta"] = (
    df_test_avaliacao["oferta_recomendada"] == df_test_avaliacao["oferta_real"]
).astype(int)

# Taxa de conversão observada quando o modelo "acertou" a oferta
taxa_modelo_acertou = df_test_avaliacao[
    df_test_avaliacao["modelo_acertou_oferta"] == 1
][TARGET].mean()

# Taxa de conversão observada quando o modelo "errou" a oferta
taxa_modelo_errou = df_test_avaliacao[
    df_test_avaliacao["modelo_acertou_oferta"] == 0
][TARGET].mean()

pct_acertou = df_test_avaliacao["modelo_acertou_oferta"].mean()

print("=" * 60)
print("ESTIMATIVA DE IMPACTO CORRIGIDA — Observado vs Observado")
print("=" * 60)
print(f"Clientes que receberam a oferta recomendada: {pct_acertou*100:.1f}%")
print(f"  → Taxa de conversão observada: {taxa_modelo_acertou*100:.1f}%")
print(f"Clientes que receberam outra oferta:          {(1-pct_acertou)*100:.1f}%")
print(f"  → Taxa de conversão observada: {taxa_modelo_errou*100:.1f}%")
print(f"\nBaseline (melhor uniforme — discount): {taxa_baseline_discount*100:.1f}%")
print(f"Ganho estimado (observado vs observado): "
      f"+{(taxa_modelo_acertou - taxa_baseline_discount)*100:.1f} p.p.")

print(f"\n{'=' * 60}")
print("CUSTO OPERACIONAL — Sure Things (Lo, 2002)")
print("=" * 60)
print(f"Sure Things: {sure_things} clientes ({pct_sure_things*100:.1f}%)")
print(f"Custo evitável estimado: R${sure_things * df_test['discount_value'].mean():.2f}")
print(f"\nNota metodológica: o threshold de {THRESHOLD_SURE_THING} é conservador.")
print(f"Em produção, esse threshold seria calibrado via experimento A/B")
print(f"pra balancear custo de cupom vs risco de perder conversão.")

In [0]:
# ============================================================
# PARTE 6.2 CORRIGIDA — IMPACTO FINANCEIRO
#
# Correção: Sure Things calculados por cliente único,
# não por linha (um cliente pode ter múltiplas linhas no teste
# — uma por instância de oferta recebida)
# ============================================================

ticket_medio = df_test["transaction_amount"].dropna().mean()
custo_medio_cupom = df_test["discount_value"].mean()
n_clientes_total = df["customer_id"].nunique()
n_clientes_teste = df_test["customer_id"].nunique()
fator_escala = n_clientes_total / n_clientes_teste
ganho_pct_conversao = taxa_modelo_acertou - taxa_baseline_discount

# Sure Things por cliente único (não por linha)
# Para cada cliente, pega a probabilidade média de conversão
# por tipo de oferta (média das suas instâncias no teste)
sure_things_por_cliente = (
    resultados_uplift
    .assign(customer_id=df_test["customer_id"].values)
    .groupby("customer_id")[["p_conv_bogo","p_conv_discount","p_conv_informational"]]
    .mean()
)

sure_things_clientes = (
    (sure_things_por_cliente["p_conv_bogo"] > THRESHOLD_SURE_THING) &
    (sure_things_por_cliente["p_conv_discount"] > THRESHOLD_SURE_THING) &
    (sure_things_por_cliente["p_conv_informational"] > THRESHOLD_SURE_THING)
).sum()

pct_sure_things_clientes = sure_things_clientes / n_clientes_teste

# Persuadables = clientes que NÃO são Sure Things
persuadables_teste = n_clientes_teste - sure_things_clientes
persuadables_total = n_clientes_total - int(sure_things_clientes * fator_escala)

# Custo evitável
custo_evitavel_teste = sure_things_clientes * custo_medio_cupom
custo_evitavel_total = int(sure_things_clientes * fator_escala) * custo_medio_cupom

# Receita incremental
clientes_adicionais_teste  = ganho_pct_conversao * n_clientes_teste
clientes_adicionais_total  = ganho_pct_conversao * n_clientes_total
receita_incremental_teste  = clientes_adicionais_teste * ticket_medio
receita_incremental_total  = clientes_adicionais_total * ticket_medio

# Custo da campanha personalizada (só pros Persuadables)
custo_campanha_teste  = persuadables_teste * custo_medio_cupom
custo_campanha_total  = persuadables_total * custo_medio_cupom

# ROI
roi_teste  = (receita_incremental_teste + custo_evitavel_teste) / custo_campanha_teste
roi_total  = (receita_incremental_total + custo_evitavel_total) / custo_campanha_total

print("=" * 60)
print("PREMISSAS DO MODELO FINANCEIRO")
print("=" * 60)
print(f"Ticket médio:                  R${ticket_medio:.2f}")
print(f"Custo médio do cupom:          R${custo_medio_cupom:.2f}")
print(f"Clientes no teste:             {n_clientes_teste:,}")
print(f"Clientes na base completa:     {n_clientes_total:,}")
print(f"Ganho de conversão observado:  +{ganho_pct_conversao*100:.1f} p.p.")
print(f"Sure Things (por cliente):     {sure_things_clientes} ({pct_sure_things_clientes*100:.1f}%)")
print(f"Persuadables (por cliente):    {persuadables_teste} ({(1-pct_sure_things_clientes)*100:.1f}%)")

print(f"\n{'=' * 60}")
print("IMPACTO FINANCEIRO — Conjunto de Teste (3.4k clientes)")
print("=" * 60)
print(f"Receita incremental:            R${receita_incremental_teste:,.2f}")
print(f"Custo evitável (Sure Things):   R${custo_evitavel_teste:,.2f}")
print(f"Custo da campanha (Persuadables): R${custo_campanha_teste:,.2f}")
print(f"ROI da personalização:          {roi_teste:.2f}x")

print(f"\n{'=' * 60}")
print("IMPACTO FINANCEIRO — Base Completa (17k clientes)")
print("=" * 60)
print(f"Receita incremental:            R${receita_incremental_total:,.2f}")
print(f"Custo evitável (Sure Things):   R${custo_evitavel_total:,.2f}")
print(f"Custo da campanha (Persuadables): R${custo_campanha_total:,.2f}")
print(f"ROI da personalização:          {roi_total:.2f}x")

# Sumário Executivo — Achados do Modelo

---

## 1. Modelo
**S-Learner LightGBM** | AUC-ROC: **0.793** | Average Precision: **0.764**

Calibração excelente — probabilidades refletem frequências reais, validando o cálculo de uplift por subtração entre tipos de oferta.

---

## 2. Drivers de Conversão (SHAP)
Top features por impacto (Gain):

| Rank | Feature | Direção | Interpretação |
|---|---|---|---|
| 1 | `dias_de_cadastro` | ↑ alto → mais conversão | Fidelidade é o melhor preditor de engajamento |
| 2 | `channel_social` | ↑ ativo → mais conversão | Canal social aumenta conversão significativamente |
| 3 | `credit_card_limit` | ↑ alto → mais conversão | Poder aquisitivo favorece completar a oferta |
| 4 | `age` | ↑ mais velho → mais conversão | Clientes 45+ convertem ~12 p.p. mais que <25 anos |
| 9 | `offer_type_encoded` | impacto marginal | **Tipo de oferta é driver secundário** |

<div style="background-color:#FFF3CD; padding:12px; border-radius:6px; margin-top:8px;">
⚠️ <b>Achado crítico:</b> O perfil do cliente importa mais que o tipo de oferta. 
A maior alavanca de ROI é <b>pra quem enviar</b>, não <b>qual oferta enviar</b>.
</div>

---

## 3. Heterogeneidade Acionável (Shchetkina & Berman, EC '24)

| Oferta | % Clientes recomendados | P(conversão) média |
|---|---|---|
| **Discount** | 78.6% | 0.703 |
| **Informational** | 21.4% | 0.614 |
| ~~BOGO~~ | 0% — nunca recomendado | 0.531 |

Há heterogeneidade acionável **parcial**: crossover real entre discount e informational existe, mas BOGO é estratégia uniformemente dominada pelo discount em todos os segmentos de cliente.

---

## 4. Impacto de Negócio (estimativa conservadora — observado vs observado)

| Métrica | Teste (3.4k clientes) | Base completa (17k clientes) |
|---|---|---|
| Ganho de conversão | +5.1 p.p. sobre baseline | +5.1 p.p. |
| Receita incremental | R$2.340 | R$11.699 |
| Custo evitável (Sure Things) | R$4.129 | R$20.638 |
| Custo da campanha (Persuadables) | R$10.279 | R$51.394 |
| ROI direto | 0.63x | 0.63x |

<div style="background-color:#FFF3CD; padding:12px; border-radius:6px; margin-top:8px;">
⚠️ <b>ROI negativo no curto prazo não invalida a estratégia:</b><br>
A plataforma retém ~10-15% do ticket médio (R$13.59) → receita real por transação incremental: R$1.36–2.04.<br>
O cupom se paga em <b>2-3 recompras subsequentes</b> se a retenção aumentar ≥ 3 p.p. — análise de LTV é o próximo passo natural.
</div>

---

## 5. Recomendações

- **Descontinuar BOGO** ou redesenhar parâmetros (`min_value` e `duration` atuais geram performance uniformemente inferior ao discount)
- **Priorizar canal social** na distribuição de ofertas (2º driver de conversão)
- **Focar cupons em clientes recentes** (baixo `dias_de_cadastro`) — maior potencial de LTV incremental, cupom como mecanismo de ativação
- **Não enviar cupom para Sure Things** (28.7% da base) — economia de **R$20.6k por rodada** de campanha na base de 17k clientes

---

## 6. Limitações e Próximos Passos

| Limitação | Próximo Passo |
|---|---|
| Sem grupo controle formal (Lo, 2002 requer A/B explícito) | Implementar experimento controlado em produção |
| ROI baseado em transação única, sem LTV | Modelo de sobrevivência / recompra (Kaplan-Meier) |
| Base pequena (17k vs milhões de usuários reais) | Replicar pipeline em escala com dados de produção |
| Hiperparâmetros não otimizados | Optuna/Bayesian search — ganho estimado de 2-3 p.p. em AUC |
| BOGO descartado com perfil médio | Análise separada com parâmetros redesenhados |

---

*Referências: Lo (2002) — ACM SIGKDD Explorations | Shchetkina & Berman (2024) — EC '24, ACM | Künzel et al. (2019) — S/T/X-Learner | Lundberg & Lee (2017) — SHAP*